# Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
sns.set(style="whitegrid", context="notebook")

from sklearn.model_selection import train_test_split
from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")

print(tf.version.VERSION)
print(keras.__version__)

2.7.0
2.7.0


# Auxilary Functions

In [ ]:
def split_input_target(sequence):
    sequence = list(sequence)
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return {"input_seq": input_text, "output_seq": target_text}

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_object_ = tf.keras.losses.SparseCategoricalCrossentropy(
      from_logits=True, 
      reduction="none"
      ) 
    loss_ = loss_object_(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

def model_train(model, train_x, train_y, callback=False,
                batch_size=128, verbose=1):

    return model.fit(train_x, train_y,
                     epochs=200,
                     callbacks=callback,
                     batch_size=batch_size,
                     validation_split=0.1,
                     verbose=verbose)
  
def model_eval(model, test_x, h_df, cust_list, k=5):
    hit = 0
    cus = 0
    mrr = 0
    retk = set()
    p = 0
    r = 0
    y_pred = model.predict(test_x)
    cust = cust_list
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = set(h_df.loc[h_df["c_id"]==c, "item_code"].values)
        if len(rel) > 0:
            ret = y_pred[i][-1].argsort()[::-1][:k]
            cus += 1
            retk = retk.union(set(ret))
            inter = set.intersection(rel, ret)
            if len(inter) > 0:
                rank = 5
                for j in inter:
                    th = np.where(ret==j)
                    if th[0][0]<=rank:
                        rank = th[0][0]  
                mrr += 1/(rank+1)
                hit += 1
                p += len(inter)/len(ret)
                r += len(inter)/len(rel)

    return p/cus, r/cus, mrr/cus, hit/cus, len(retk)

def model_graph(hist):
    plt.plot(hist.history["loss"])
    plt.plot(hist.history["val_loss"])
    plt.title(hist.model.name+" loss")
    plt.ylabel("loss")
    plt.xlabel("epoch")
    plt.legend(["train", "val"], loc="best")
    plt.show()

    plt.plot(hist.history["sparse_categorical_accuracy"])
    plt.plot(hist.history["val_sparse_categorical_accuracy"])
    plt.title(hist.model.name+" accuracy")
    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.legend(["train", "val"], loc="best")
    plt.show()

def create_sequences(df):
    df["item_code"] = df["item_code"].astype("string")
    df["recency"] = df["recency"].astype("string")
    df["payment"] = df["payment"].astype("string")
    df["category_code"] = df["category_code"].astype("string")
    df["method_code"] = df["method_code"].astype("string")
    g_df = df.groupby("c_id", as_index=False).agg(
          {"item_code": lambda x: ", ".join(x),
          "recency": lambda y: ", ".join(y),
          "payment": lambda z: ", ".join(z),
          "category_code": lambda t: ", ".join(t),
          "method_code": lambda u: ", ".join(u),
          "t_id":"count"}
          )
    g_df["item_code"] = g_df["item_code"].apply(
      lambda x: x.split(", ")
      )
    g_df["recency"] = g_df["recency"].apply(
      lambda y: y.split(", ")
      )
    g_df["payment"] = g_df["payment"].apply(
      lambda z: z.split(", ")
      )
    g_df["category_code"] = g_df["category_code"].apply(
      lambda t: t.split(", ")
      )
    g_df["method_code"] = g_df["method_code"].apply(
      lambda u: u.split(", ")
      )
    return g_df

def create_splits(g_df):
    s_df = g_df.merge(g_df["item_code"].apply(
      lambda w: pd.Series(split_input_target(w))
      ), left_index=True, right_index=True)
    s_df["recency"] = s_df["recency"].apply(lambda p: list(p)[:-1])
    s_df["payment"] = s_df["payment"].apply(lambda q: list(q)[:-1])
    s_df["category_code"] = s_df["category_code"].apply(lambda r: list(r)[:-1])
    s_df["method_code"] = s_df["method_code"].apply(lambda s: list(s)[:-1])
    return s_df

def make_padding(df, n_padded):
    input_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["input_seq"].values, 
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    recency_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["recency"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    payment_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["payment"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    category_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["category_code"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    method_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["method_code"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    if len(df.columns[df.columns.str.contains("output_seq")]) > 0:
        output_seq_padded = keras.preprocessing.sequence.pad_sequences(
            df["output_seq"].values,
            maxlen=n_padded,
            padding="pre",
            value=0.0
        )
    else:
        output_seq_padded = []
    return [input_seq_padded,
            recency_seq_padded,
            payment_seq_padded,
            category_seq_padded,
            method_seq_padded,
            output_seq_padded]

# Data Processing

In [ ]:
!wget "https://github.com/ibrahimerdem/application_storage/raw/main/ecommerce_dataset.zip"
!unzip "ecommerce_dataset.zip"
data = pd.read_csv("ecommerce_dataset.csv")

--2021-11-21 20:30:34--  https://github.com/ibrahimerdem/application_storage/raw/main/ecommerce_dataset.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/ecommerce_dataset.zip [following]
--2021-11-21 20:30:34--  https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/ecommerce_dataset.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14653198 (14M) [application/zip]
Saving to: ‘ecommerce_dataset.zip’

ecommerce_dataset.z 100%[===================>]  13.97M  81.0MB/s    in 0.2s    

2021-11-21 20:30:34 (81.0 MB/s) - ‘ecommerce_dataset.zip’

In [ ]:
df = data.copy()

## Filtering

In [ ]:
df = df[df["status"]=="complete"]
df = df.iloc[:, [0, 2, 20, 3, 8, 4, 5, 11]]
df.rename(columns={"item_id":"t_id",
                   "created_at":"t_date",
                   "Customer ID":"c_id",
                   "sku":"item_name",
                   "category_name_1":"item_category",
                   "qty_ordered":"amount"}, inplace=True)
df["t_date"] = df["t_date"].astype("datetime64")

c_freq = 2
p_freq = 2
m_freq = 1000
t_freq = 2500
model_start = "04-01-2017"
model_end = "04-01-2018"
test_start = "04-01-2018"
test_end = "07-01-2018"

d_base = df[(df["t_date"]>=model_start)&(df["t_date"]<model_end)]

freq = pd.DataFrame(d_base["item_name"].value_counts())
f_items = freq[freq["item_name"]>p_freq].index
d_base = d_base[d_base["item_name"].isin(f_items)]
print("after first filter")
print("#customers:", d_base["c_id"].nunique(),
      "#items", d_base["item_name"].nunique())

c_data = d_base.groupby("c_id", as_index=False).agg({"t_date":"count"})
f_cust = c_data.loc[c_data["t_date"]>c_freq, "c_id"].values
d_base = d_base[d_base["c_id"].isin(f_cust)]

fmet = pd.DataFrame(d_base["payment_method"].value_counts())
f_met = fmet.loc[fmet["payment_method"]<m_freq, ["payment_method"]]
d_base.loc[d_base["payment_method"].isin(f_met.index), "payment_method"]="other"

fcat = pd.DataFrame(d_base["item_category"].value_counts())
f_cat = fcat.loc[fcat["item_category"]<t_freq, ["item_category"]]
d_base.loc[d_base["item_category"].isin(f_cat.index), "item_category"]="other"

print("\nafter second filter")
mind = d_base["t_date"].min()
maxd = d_base["t_date"].max()
print("#customers:", d_base["c_id"].nunique(),
      "#items", d_base["item_name"].nunique())

print("\ndate between:", mind.day, mind.month, mind.year,
      "-", maxd.day, maxd.month, maxd.year)
print("#transactions:", d_base["t_id"].nunique())
print("#product-categories:", d_base["item_category"].nunique())
print("#payment-methods:", d_base["payment_method"].nunique())

after first filter
#customers: 35030 #items 7909

after second filter
#customers: 10050 #items 7481

date between: 1 4 2017 - 31 3 2018
#transactions: 69254
#product-categories: 10
#payment-methods: 7


## Feature Extraction

In [ ]:
d_base["recency"] = d_base["t_date"].max() - d_base["t_date"]
d_base["recency"] = d_base["recency"].dt.days

d_base.loc[d_base["price"]==0, "price"] = 0.1
d_base["payment"] = d_base["price"]*d_base["amount"]
d_base.drop(columns=["price", "amount"], inplace=True)

d_base["recency"] = ((d_base["recency"]*99)/d_base["recency"].max())+1
d_base["recency"] = d_base["recency"].astype("int64")

d_base["payment"] = ((d_base["payment"]*999)/d_base["payment"].max())+1
d_base["payment"] = d_base["payment"].astype("int64")

## Encoding

In [ ]:
cust = d_base["c_id"].unique()
c_size = len(cust)
cust2code = {}
code2cust = {}
for num, c in enumerate(cust):
    cust2code[c] = num+1
    code2cust[num+1] = c

d_base["c_id"] = d_base["c_id"].map(cust2code)

items = d_base["item_name"].unique()
i_size = len(items)
item2code = {}
code2item = {}
for num, item in enumerate(items):
    item2code[item] = num+1
    code2item[num+1] = item

d_base["item_code"] = d_base["item_name"].map(item2code)

cats = d_base["item_category"].unique()
c_size = len(cats)
cat2code = {}
code2cat = {}
for num, cat in enumerate(cats):
    cat2code[cat] = num+1
    code2cat[num+1] = cat

d_base["category_code"] = d_base["item_category"].map(cat2code)

methods = d_base["payment_method"].unique()
m_size = len(methods)
method2code = {}
code2method = {}
for num, method in enumerate(methods):
    method2code[method] = num+1
    code2method[num+1] = method

d_base["method_code"] = d_base["payment_method"].map(method2code)
d_base = d_base.drop(columns=["item_name", "item_category", "payment_method"])

time_hold = df.loc[(df["t_date"]>=test_start)&(df["t_date"]<test_end),
                   ["c_id", "item_name"]]
time_hold["c_id"] = time_hold["c_id"].map(cust2code)
time_hold["item_code"] = time_hold["item_name"].map(item2code)
time_hold = time_hold.dropna()

## Sampling

In [ ]:
hold_c = time_hold["c_id"].unique()

time_train = d_base[~(d_base["c_id"].isin(hold_c))]
time_test = d_base[d_base["c_id"].isin(hold_c)]

print("train customers:", time_train["c_id"].nunique())
print("test customers:", time_test["c_id"].nunique())
print("held customers:", time_hold["c_id"].nunique())

train customers: 9751
test customers: 299
held customers: 299


In [ ]:
time_train.to_csv(
    "time_train_data.csv",
    sep=";",
    index=False)

time_test.to_csv(
    "time_test_data.csv",
    sep=";",
    index=False)

time_hold.to_csv(
    "time_hold_data.csv",
    sep=";",
    index=False)

# Modeling

## Preparing

In [ ]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/time_train_data.csv",
     sep=";")

tdata = pd.read_csv(
    "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/time_test_data.csv",
     sep=";")

hdata = pd.read_csv(
    "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/time_hold_data.csv",
     sep=";")

In [ ]:
df = data.copy()
t_df = tdata.copy()
h_df = hdata.copy()

In [ ]:
df_a = df.append(t_df, ignore_index=True)
pro_max = df_a["item_code"].astype("int64").max()
rec_max = df_a["recency"].astype("int64").max()
pay_max = df_a["payment"].astype("int64").max()
cat_max = df_a["category_code"].astype("int64").max()
met_max = df_a["method_code"].astype("int64").max()

In [ ]:
train_data = create_sequences(df)
train_data = create_splits(train_data)

test_data = create_sequences(t_df)
test_data.rename(columns={"item_code":"input_seq"}, inplace=True)

In [ ]:
train_data["t_id"].describe()

count    9751.000000
mean        6.514511
std         8.504320
min         3.000000
25%         3.000000
50%         4.000000
75%         7.000000
max       288.000000
Name: t_id, dtype: float64

In [ ]:
n_padded = 12
train_list = make_padding(train_data, n_padded)
test_list = make_padding(test_data, n_padded)

## Model with Item-Only Sequence

In [ ]:
def model1(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                        name="item_input",
                        dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"], 
                                    output_dim=conf["emb_units"],
                                    name="item_embedding"
                                    )(input_item)
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(embedding_item)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_1 = keras.Model(input_item, output, name="model_1")
    model_1.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_1

## Model with Item and Recency Sequences

In [ ]:
def model2(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    concat_inputs = layers.Concatenate(name="concat_inputs")(
                [embedding_item, embedding_rec])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_2 = keras.Model([input_item, input_rec], output, name="model_2")
    model_2.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_2

## Model with Item, Recency and Payment Sequences

In [ ]:
def model3(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    input_pay = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="pay_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    embedding_pay = layers.Embedding(input_dim=conf["pay_max"],
                                     output_dim=conf["emb_units"],
                                     name="pay_embedding"
                                     )(input_pay)
    concat_inputs = layers.Concatenate(name="concat_inputs")(
                [embedding_item, embedding_rec, embedding_pay])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_3 = keras.Model([input_item,
                           input_rec,
                           input_pay], output, name="model_3")
    model_3.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_3

## Model with Item, Recency and Category Sequences

In [ ]:
def model4(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="cat_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    embedding_cat = layers.Embedding(input_dim=conf["cat_max"],
                                     output_dim=conf["emb_units"],
                                     name="cat_embedding"
                                     )(input_cat)
    concat_inputs = layers.Concatenate(name="concat_inputs")(
                [embedding_item, embedding_rec, embedding_cat])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_4 = keras.Model([input_item,
                           input_rec,
                           input_cat], output, name="model_4")
    model_4.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_4


## Model with Item, Recency and Payment-Method Sequences

In [ ]:
def model5(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    input_met = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="met_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    embedding_met = layers.Embedding(input_dim=conf["met_max"],
                                     output_dim=conf["emb_units"],
                                     name="met_embedding"
                                     )(input_met)
    concat_inputs = layers.Concatenate(name="concat_inputs")(
                [embedding_item, embedding_rec, embedding_met])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_5 = keras.Model([input_item,
                           input_rec,
                           input_met], output, name="model_5")
    model_5.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_5

## Model with Item, Recency, Payment and Category Sequences

In [ ]:
def model6(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    input_pay = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="pay_input",
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="cat_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    embedding_pay = layers.Embedding(input_dim=conf["pay_max"],
                                     output_dim=conf["emb_units"],
                                     name="pay_embedding"
                                     )(input_pay)
    embedding_cat = layers.Embedding(input_dim=conf["cat_max"],
                                     output_dim=conf["emb_units"],
                                     name="cat_embedding"
                                     )(input_cat)                                
    concat_inputs = layers.Concatenate(name="concat_inputs")(
        [embedding_item, embedding_rec, embedding_pay, embedding_cat])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_6 = keras.Model([input_item,
                           input_rec,
                           input_pay,
                           input_cat], output, name="model_6")
    model_6.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_6

## Model with Item, Recency, Category and Payment-Method Sequences

In [ ]:
def model7(conf):
    input_item = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                              name="item_input",
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="rec_input",
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="cat_input",
                             dtype=tf.int32)
    input_met = layers.Input(batch_input_shape=[None, conf["n_padded"]],
                             name="met_input",
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=conf["item_max"],
                                      output_dim=conf["emb_units"],
                                      name="item_embedding"
                                      )(input_item)
    embedding_rec = layers.Embedding(input_dim=conf["rec_max"],
                                     output_dim=conf["emb_units"],
                                     name="rec_embedding"
                                     )(input_rec)
    embedding_cat = layers.Embedding(input_dim=conf["cat_max"],
                                     output_dim=conf["emb_units"],
                                     name="cat_embedding"
                                     )(input_cat)      
    embedding_met = layers.Embedding(input_dim=conf["met_max"],
                                     output_dim=conf["emb_units"],
                                     name="met_embedding"
                                     )(input_met)                                                         
    concat_inputs = layers.Concatenate(name="concat_inputs")([embedding_item,
                                                              embedding_rec,
                                                              embedding_cat,
                                                              embedding_met])
    concat_inputs = layers.BatchNormalization(
             name="batchnorm_inputs")(concat_inputs)                             
    lstm = layers.LSTM(units=conf["lstm_units"],
                    return_sequences=True,
                    name="lstm_layer"
                    )(concat_inputs)
    lstm = layers.Dropout(conf["drop_rate"])(lstm)
    lstm = layers.BatchNormalization(name="batchnorm_lstm")(lstm)
    att = layers.Attention(use_scale=False,
                           causal=True,
                           name="attention")(inputs=[lstm, lstm],
                                             mask=[encoding_padding_mask,
                                                   encoding_padding_mask])
    output = layers.Dense(conf["item_max"], name="output")(att)
    model_7 = keras.Model([input_item,
                           input_rec,
                           input_cat,
                           input_met], output, name="model_7")
    model_7.compile(loss=conf["loss"],
                    optimizer=conf["optimizer"],
                    metrics=[conf["metric"]])
    return model_7

## Training Lines

In [ ]:
es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                   mode="min",
                                   verbose=1,
                                   patience=3)

conf = {
    "n_padded":n_padded,
    "item_max":pro_max+1,
    "rec_max":rec_max+1,
    "pay_max":pay_max+1,
    "cat_max":cat_max+1,
    "met_max":met_max+1,
    "emb_units":96,
    "lstm_units":100,
    "drop_rate":0.8,
    "loss":loss_function,
    "optimizer":keras.optimizers.Adam(learning_rate=0.001),
    "metric":"sparse_categorical_accuracy"}

### Model-1

In [ ]:
model_1 = model1(conf)

hist_1 = model_train(model_1, train_list[0], train_list[-1],
                     callback=[es], batch_size=128)

Epoch 1/200
69/69 [==============================] - 11s 40ms/step - loss: 1.2247 - sparse_categorical_accuracy: 0.0432 - val_loss: 0.7691 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 2/200
69/69 [==============================] - 2s 27ms/step - loss: 1.0840 - sparse_categorical_accuracy: 0.0573 - val_loss: 0.7296 - val_sparse_categorical_accuracy: 0.0026
Epoch 3/200
69/69 [==============================] - 2s 27ms/step - loss: 1.0479 - sparse_categorical_accuracy: 0.0767 - val_loss: 0.7056 - val_sparse_categorical_accuracy: 0.0163
Epoch 4/200
69/69 [==============================] - 2s 28ms/step - loss: 1.0190 - sparse_categorical_accuracy: 0.0963 - val_loss: 0.6815 - val_sparse_categorical_accuracy: 0.0486
Epoch 5/200
69/69 [==============================] - 2s 28ms/step - loss: 0.9889 - sparse_categorical_accuracy: 0.1170 - val_loss: 0.6484 - val_sparse_categorical_accuracy: 0.1388
Epoch 6/200
69/69 [==============================] - 2s 27ms/step - loss: 0.9540 - sparse_categ

### Model-2

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:128

model_2 = model2(conf)

hist_2 = model_train(model_2,
                     [train_list[0], train_list[1]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 39ms/step - loss: 1.4874 - sparse_categorical_accuracy: 0.0672 - val_loss: 1.0108 - val_sparse_categorical_accuracy: 0.0020
Epoch 2/200
69/69 [==============================] - 2s 27ms/step - loss: 1.3483 - sparse_categorical_accuracy: 0.1089 - val_loss: 0.9871 - val_sparse_categorical_accuracy: 0.0125
Epoch 3/200
69/69 [==============================] - 2s 27ms/step - loss: 1.2881 - sparse_categorical_accuracy: 0.1341 - val_loss: 0.9437 - val_sparse_categorical_accuracy: 0.0743
Epoch 4/200
69/69 [==============================] - 2s 27ms/step - loss: 1.2421 - sparse_categorical_accuracy: 0.1487 - val_loss: 0.8776 - val_sparse_categorical_accuracy: 0.2157
Epoch 5/200
69/69 [==============================] - 2s 27ms/step - loss: 1.2054 - sparse_categorical_accuracy: 0.1578 - val_loss: 0.7815 - val_sparse_categorical_accuracy: 0.2552
Epoch 6/200
69/69 [==============================] - 2s 26ms/step - loss: 1.1720 - sparse_categorica

### Model-3

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:128

model_3 = model3(conf)

hist_3 = model_train(model_3,
                     [train_list[0], train_list[1], train_list[2]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 40ms/step - loss: 1.4780 - sparse_categorical_accuracy: 0.0745 - val_loss: 0.9993 - val_sparse_categorical_accuracy: 0.0142
Epoch 2/200
69/69 [==============================] - 2s 29ms/step - loss: 1.3476 - sparse_categorical_accuracy: 0.1138 - val_loss: 0.9829 - val_sparse_categorical_accuracy: 0.0180
Epoch 3/200
69/69 [==============================] - 2s 29ms/step - loss: 1.2965 - sparse_categorical_accuracy: 0.1344 - val_loss: 0.9189 - val_sparse_categorical_accuracy: 0.1120
Epoch 4/200
69/69 [==============================] - 2s 29ms/step - loss: 1.2548 - sparse_categorical_accuracy: 0.1462 - val_loss: 0.8529 - val_sparse_categorical_accuracy: 0.2017
Epoch 5/200
69/69 [==============================] - 2s 29ms/step - loss: 1.2197 - sparse_categorical_accuracy: 0.1543 - val_loss: 0.7648 - val_sparse_categorical_accuracy: 0.2607
Epoch 6/200
69/69 [==============================] - 2s 28ms/step - loss: 1.1908 - sparse_categorica

### Model-4

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:400

model_4 = model4(conf)

hist_4 = model_train(model_4,
                     [train_list[0], train_list[1], train_list[3]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 42ms/step - loss: 1.1238 - sparse_categorical_accuracy: 0.0649 - val_loss: 0.7296 - val_sparse_categorical_accuracy: 0.0029
Epoch 2/200
69/69 [==============================] - 2s 30ms/step - loss: 1.0199 - sparse_categorical_accuracy: 0.1055 - val_loss: 0.7208 - val_sparse_categorical_accuracy: 0.0057
Epoch 3/200
69/69 [==============================] - 2s 30ms/step - loss: 0.9718 - sparse_categorical_accuracy: 0.1275 - val_loss: 0.6924 - val_sparse_categorical_accuracy: 0.0254
Epoch 4/200
69/69 [==============================] - 2s 29ms/step - loss: 0.9375 - sparse_categorical_accuracy: 0.1412 - val_loss: 0.6432 - val_sparse_categorical_accuracy: 0.1320
Epoch 5/200
69/69 [==============================] - 2s 30ms/step - loss: 0.9099 - sparse_categorical_accuracy: 0.1507 - val_loss: 0.5743 - val_sparse_categorical_accuracy: 0.2428
Epoch 6/200
69/69 [==============================] - 2s 31ms/step - loss: 0.8869 - sparse_categorica

### Model-5

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:128

model_5 = model5(conf)

hist_5 = model_train(model_5,
                     [train_list[0], train_list[1], train_list[4]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 41ms/step - loss: 1.4855 - sparse_categorical_accuracy: 0.0614 - val_loss: 1.0069 - val_sparse_categorical_accuracy: 0.0075
Epoch 2/200
69/69 [==============================] - 2s 29ms/step - loss: 1.3588 - sparse_categorical_accuracy: 0.0981 - val_loss: 0.9906 - val_sparse_categorical_accuracy: 0.0110
Epoch 3/200
69/69 [==============================] - 2s 29ms/step - loss: 1.3073 - sparse_categorical_accuracy: 0.1168 - val_loss: 0.9342 - val_sparse_categorical_accuracy: 0.0401
Epoch 4/200
69/69 [==============================] - 2s 28ms/step - loss: 1.2680 - sparse_categorical_accuracy: 0.1316 - val_loss: 0.8781 - val_sparse_categorical_accuracy: 0.0972
Epoch 5/200
69/69 [==============================] - 2s 28ms/step - loss: 1.2319 - sparse_categorical_accuracy: 0.1424 - val_loss: 0.7976 - val_sparse_categorical_accuracy: 0.1806
Epoch 6/200
69/69 [==============================] - 2s 29ms/step - loss: 1.2036 - sparse_categorica

### Model-6

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:600

model_6 = model6(conf)

hist_6 = model_train(model_6,
                     [train_list[0], train_list[1],
                      train_list[2], train_list[3]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 44ms/step - loss: 1.1194 - sparse_categorical_accuracy: 0.0708 - val_loss: 0.7334 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 2/200
69/69 [==============================] - 2s 31ms/step - loss: 1.0215 - sparse_categorical_accuracy: 0.1087 - val_loss: 0.6930 - val_sparse_categorical_accuracy: 0.0257
Epoch 3/200
69/69 [==============================] - 2s 31ms/step - loss: 0.9810 - sparse_categorical_accuracy: 0.1291 - val_loss: 0.6603 - val_sparse_categorical_accuracy: 0.0911
Epoch 4/200
69/69 [==============================] - 2s 31ms/step - loss: 0.9494 - sparse_categorical_accuracy: 0.1419 - val_loss: 0.5958 - val_sparse_categorical_accuracy: 0.1994
Epoch 5/200
69/69 [==============================] - 2s 32ms/step - loss: 0.9234 - sparse_categorical_accuracy: 0.1497 - val_loss: 0.5460 - val_sparse_categorical_accuracy: 0.2548
Epoch 6/200
69/69 [==============================] - 2s 31ms/step - loss: 0.9002 - sparse_catego

### Model-7

In [ ]:
conf["emb_units"]:96
conf["lstm_units"]:128

model_7 = model7(conf)

hist_7 = model_train(model_7,
                     [train_list[0], train_list[1],
                      train_list[3], train_list[4]],
                     train_list[-1],
                     callback=[es],
                     batch_size=128)

Epoch 1/200
69/69 [==============================] - 6s 43ms/step - loss: 1.4781 - sparse_categorical_accuracy: 0.0663 - val_loss: 1.0112 - val_sparse_categorical_accuracy: 0.0023
Epoch 2/200
69/69 [==============================] - 2s 30ms/step - loss: 1.3430 - sparse_categorical_accuracy: 0.1014 - val_loss: 0.9708 - val_sparse_categorical_accuracy: 0.0148
Epoch 3/200
69/69 [==============================] - 2s 30ms/step - loss: 1.2922 - sparse_categorical_accuracy: 0.1185 - val_loss: 0.9184 - val_sparse_categorical_accuracy: 0.0795
Epoch 4/200
69/69 [==============================] - 2s 30ms/step - loss: 1.2512 - sparse_categorical_accuracy: 0.1322 - val_loss: 0.8273 - val_sparse_categorical_accuracy: 0.1945
Epoch 5/200
69/69 [==============================] - 2s 30ms/step - loss: 1.2143 - sparse_categorical_accuracy: 0.1421 - val_loss: 0.7523 - val_sparse_categorical_accuracy: 0.2157
Epoch 6/200
69/69 [==============================] - 2s 30ms/step - loss: 1.1849 - sparse_categorica

## Evaluation

### Graphs

In [ ]:
model_graph(hist_1)
model_graph(hist_2)
model_graph(hist_3)
model_graph(hist_4)
model_graph(hist_5)
model_graph(hist_6)

### Metrics

In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_1.model,
                                      [test_list[0]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1204
Precision@5 : 0.0301
Recall@5 : 0.0731
F1@5 : 0.0426
MRR@5 : 0.0691
Cov@5 : 0.0793


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_2.model,
                                      [test_list[0], test_list[1]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1271
Precision@5 : 0.0294
Recall@5 : 0.0841
F1@5 : 0.0436
MRR@5 : 0.0775
Cov@5 : 0.058


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_3.model,
                                      [test_list[0], test_list[1],
                                       test_list[2]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1472
Precision@5 : 0.0314
Recall@5 : 0.0918
F1@5 : 0.0468
MRR@5 : 0.0831
Cov@5 : 0.0636


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_4.model,
                                      [test_list[0], test_list[1],
                                       test_list[3]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1739
Precision@5 : 0.0375
Recall@5 : 0.1153
F1@5 : 0.0565
MRR@5 : 0.0984
Cov@5 : 0.0636


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_5.model,
                                      [test_list[0], test_list[1],
                                       test_list[4]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1405
Precision@5 : 0.0301
Recall@5 : 0.0927
F1@5 : 0.0454
MRR@5 : 0.0811
Cov@5 : 0.0595


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_6.model,
                                      [test_list[0], test_list[1],
                                       test_list[2], test_list[3]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1538
Precision@5 : 0.0368
Recall@5 : 0.1003
F1@5 : 0.0538
MRR@5 : 0.0907
Cov@5 : 0.0611


In [ ]:
k = 5
pk, rk, mrrk, hitk, retk = model_eval(hist_7.model,
                                      [test_list[0], test_list[1],
                                       test_list[3], test_list[4]],
                                      h_df,
                                      test_data["c_id"].unique(),
                                      k)

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(retk/pro_max, 4)}")

Hit@5 : 0.1706
Precision@5 : 0.0381
Recall@5 : 0.1141
F1@5 : 0.0572
MRR@5 : 0.0922
Cov@5 : 0.0577


### Baselines

#### Popular k

In [ ]:
k = 5
cus = 0
hit = 0
mrr = 0
p = 0
r = 0
y_pred = [int(y) for y in df["item_code"].value_counts().head(k).keys()]
cust = test_data["c_id"].unique()
counter = len(cust)
for i in range(counter):
    c = cust[i]
    rel = set(h_df.loc[h_df["c_id"]==c, "item_code"].values)
    if len(rel) > 0:
        ret = np.array(y_pred)
        cus += 1
        inter = set.intersection(rel, ret)
        if len(inter) > 0:
            rank = 1000
            for j in inter:
                th = np.where(ret==j)
                if th[0][0]<=rank:
                    rank = th[0][0]    
            mrr += 1/(rank+1)
            hit += 1
            p += len(inter)/len(ret)
            r += len(inter)/len(rel)

pk = p/cus
rk = r/cus
mrrk = mrr/cus
hitk = hit/cus

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(k/pro_max, 4)}")

Hit@5 : 0.0517
Precision@5 : 0.0103
Recall@5 : 0.0409
F1@5 : 0.0165
MRR@5 : 0.0129
Cov@5 : 0.0004


#### Last k

In [ ]:
k = 5
cus = 0
hit = 0
mrr = 0
retk = set()
p = 0
r = 0
cust = test_data["c_id"].unique()
counter = len(cust)
for i in range(counter):
    c = cust[i]
    rel = set(h_df.loc[h_df["c_id"]==c, "item_code"].values)
    if len(rel) > 0:
        re = test_data.loc[test_data["c_id"]==c, "input_seq"]
        ret = np.array(list(set([int(z) for z in re.values[0][:][::-1]])))[:k]
        if len(ret)==k:
            cus += 1
            retk = retk.union(set(ret))
            inter = set.intersection(rel, ret)
            if len(inter) > 0:
                rank = 1000
                for j in inter:
                    th = np.where(ret==j)
                    if th[0][0]<=rank:
                        rank = th[0][0]    
                mrr += 1/(rank+1)
                hit += 1
                p += len(inter)/len(ret)
                r += len(inter)/len(rel)

pk = p/cus
rk = r/cus
mrrk = mrr/cus
hitk = hit/cus

print(f"Hit@{k} : {np.round(hitk, 4)}")
print(f"Precision@{k} : {np.round(pk, 4)}")
print(f"Recall@{k} : {np.round(rk, 4)}")
print(f"F1@{k} : {np.round((2*pk*rk)/(pk+rk),4)}")
print(f"MRR@{k} : {np.round(mrrk, 4)}")
print(f"Cov@{k} : {np.round(len(retk)/pro_max, 4)}")

Hit@5 : 0.0766
Precision@5 : 0.0204
Recall@5 : 0.0529
F1@5 : 0.0295
MRR@5 : 0.0357
Cov@5 : 0.0791


# Experimental Results

[Parameter Tuning](https://colab.research.google.com/drive/1P8WpMI17KgnLghIQc5sLvEwbQ3Sub8s5#scrollTo=5b7088a5)

[Application Experiments](https://colab.research.google.com/drive/1eZ1R2sSYp8x5iEP7VKD7GuMGJEDW67qh#scrollTo=91fb701d)

[Results](https://colab.research.google.com/drive/1KkV7vAa4K_py6njt6Ci2mUHMgqXWHhlD#scrollTo=DCkWne92Mz_c&line=1&uniqifier=1)